In [ ]:
# SAE Hooks Verification Notebook

import torch as t
from transformers import AutoTokenizer
from sae_lens import HookedSAETransformer, SAE

t.set_grad_enabled(False)

# Load model and SAE
def load_model_and_sae(layer, device):
    model_name = "google/gemma-2-2b-it"
    model = HookedSAETransformer.from_pretrained(model_name, device=device)
    sae, _, _ = SAE.from_pretrained(
        release="gemma-scope-2b-pt-res-canonical",
        sae_id=f"layer_{layer}/width_16k/canonical",
        device=str(device),
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    sae.use_error_term = False
    return model, sae, tokenizer

# Setup

layer = 5
device = t.device("cuda" if t.cuda.is_available() else "cpu")
model, sae, tokenizer = load_model_and_sae(layer, device)





Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b-it into HookedTransformer


In [6]:
def do_Assertions():
    # create assertions for the ablation results, to check if out and recon tensors are similar to each other, or to l5_acts or l5_acts_recon after latent manipulation
    print(f"sae.use_error_term = {sae.use_error_term}")

    # Ablation assertions
    print("\n=== ABLATION ASSERTIONS ===")

    # Assertion 1: Input should be the same for both cases
    print("Assertion 1: Input is same for both cases")
    try:
        # Check shapes first
        if cache_no_ablation['in'].shape != cache_with_ablation['in'].shape:
            print(f"❌ SHAPE MISMATCH: cache_no_ablation['in'].shape = {cache_no_ablation['in'].shape}, cache_with_ablation['in'].shape = {cache_with_ablation['in'].shape}")
        else:
            print(f"✅ Shapes match: {cache_no_ablation['in'].shape}")
        
        assert t.allclose(cache_no_ablation['in'], cache_with_ablation['in'], atol=1e-3)
        # assert t.allclose(l5_acts, cache_no_ablation['in'], atol=1e-3)
        print("✅ PASSED")
    except AssertionError:
        print("❌ FAILED")

    # Assertion 2: Features should be same for both cases
    print("\nAssertion 2: Features are same for both cases")
    try:
        # Check shapes first
        if cache_no_ablation['feats'].shape != cache_with_ablation['feats'].shape:
            print(f"❌ SHAPE MISMATCH: cache_no_ablation['feats'].shape = {cache_no_ablation['feats'].shape}, cache_with_ablation['feats'].shape = {cache_with_ablation['feats'].shape}")
        else:
            print(f"✅ Shapes match: {cache_no_ablation['feats'].shape}")
            
        assert t.allclose(cache_no_ablation['feats'], cache_with_ablation['feats'], atol=1e-3)
        # assert t.allclose(l5_latents, cache_no_ablation['feats'], atol=1e-3)
        print("✅ PASSED")
    except AssertionError:
        print("❌ FAILED")

    # Assertion 3: Reconstruction should be same
    print("\nAssertion 3: Reconstruction is same for both cases")
    try:
        # Check shapes first
        if cache_no_ablation['recon'].shape != cache_with_ablation['recon'].shape:
            print(f"❌ SHAPE MISMATCH: cache_no_ablation['recon'].shape = {cache_no_ablation['recon'].shape}, cache_with_ablation['recon'].shape = {cache_with_ablation['recon'].shape}")
        else:
            print(f"✅ Shapes match: {cache_no_ablation['recon'].shape}")
            
        assert t.allclose(cache_no_ablation['recon'], cache_with_ablation['recon'], atol=1e-3)
        print("✅ PASSED")
    except AssertionError:
        print("❌ FAILED")
        
    # Assertion 4: Output should be the same
    print("\nAssertion 4: Output is same for both cases")
    try:
        # Check shapes first
        if cache_no_ablation['out'].shape != cache_with_ablation['out'].shape:
            print(f"❌ SHAPE MISMATCH: cache_no_ablation['out'].shape = {cache_no_ablation['out'].shape}, cache_with_ablation['out'].shape = {cache_with_ablation['out'].shape}")
        else:
            print(f"✅ Shapes match: {cache_no_ablation['out'].shape}")
            
        assert t.allclose(cache_no_ablation['out'], cache_with_ablation['out'], atol=1e-3)
        print("✅ PASSED")
    except AssertionError:
        print("❌ FAILED")

    # Assertion 5: cache_with_ablation['out'] is the same as cache_with_ablation['recon']
    print("\nAssertion 5: Output is same as reconstruction for ablation case")
    try:
        # Check shapes first
        if cache_with_ablation['out'].shape != cache_with_ablation['recon'].shape:
            print(f"❌ SHAPE MISMATCH: cache_with_ablation['out'].shape = {cache_with_ablation['out'].shape}, cache_with_ablation['recon'].shape = {cache_with_ablation['recon'].shape}")
        else:
            print(f"✅ Shapes match: {cache_with_ablation['out'].shape}")
            
        assert t.allclose(cache_with_ablation['out'], cache_with_ablation['recon'], atol=1e-3)
        print("✅ PASSED")
    except AssertionError:
        print("❌ FAILED")


In [7]:
hook_base = sae.cfg.hook_name
hook_sae_input = f"{hook_base}.hook_sae_input"
hook_sae_acts_pre = f"{hook_base}.hook_sae_acts_pre"
hook_sae_acts_post = f"{hook_base}.hook_sae_acts_post"
hook_sae_output = f"{hook_base}.hook_sae_output"
hook_sae_recons = f"{hook_base}.hook_sae_recons"
hook_sae_error = f"{hook_base}.hook_sae_error"

prompt = "15+20= "
# Tokenize
tokens = tokenizer([prompt], return_tensors="pt", add_special_tokens=True).to(device)
print(f"Tokenized shape: {tokens.input_ids.shape}")

# =============================================================================
# ABLATION HOOKS AND TESTING
# =============================================================================

print("\n" + "="*50)
print("ABLATION TESTING")
print("="*50)

# Reset model state to ensure clean run
model.reset_hooks(including_permanent=True)
model.reset_saes()
sae.use_error_term = False
l5_acts = model.run_with_cache(tokens.input_ids)[1][f"blocks.{layer}.hook_resid_post"]
print(f"l5_acts shape: {l5_acts.shape}")

# Ablation parameters
lambda_scale = 5.0
ablation_features = [7792, 3056]  # important feature indices for addition in layer 5: 7792 3056
start_position = -1
end_position = None  # Ablate only the last token

# Cache for ablation runs
cache_no_ablation = {}
cache_with_ablation = {}
cache_with_zero_ablation = {}
# Ablation hook functions
def cache_input_hook_no_abl(x, hook):
    cache_no_ablation['in'] = x.clone().detach()
    return x

def cache_acts_hook_no_abl(feats, hook):
    cache_no_ablation['feats'] = feats.clone().detach()
    return feats

def cache_recon_hook_no_abl(recon, hook):
    cache_no_ablation['recon'] = recon.clone().detach()
    return recon

def cache_output_hook_no_abl(out, hook):
    cache_no_ablation['out'] = out.clone().detach()
    return out



def cache_input_hook_with_abl(x, hook):
    cache_with_ablation['in'] = x.clone().detach()
    return x

def cache_acts_hook_with_abl(feats, hook):
    cache_with_ablation['feats'] = feats.clone().detach()
    mask = t.zeros_like(feats)
    mask[:, start_position:end_position, ablation_features] = 1.0
    cache_with_ablation['feats_ablated'] = feats * mask
    return feats * mask

def cache_output_hook_with_abl(out, hook):
    cache_with_ablation['out'] = out.clone().detach()
    
    acts_out = cache_with_ablation['in'].clone()
    # initialize a tensor with the same shape as cache_with_ablation['in']
    # acts_out = t.zeros_like(cache_with_ablation['in'])
    acts_out[:, start_position:end_position, :] = (
        cache_with_ablation['in'][:, start_position:end_position, :] - 
        lambda_scale * out[:, start_position:end_position, :]
    )
    cache_with_ablation['out_ablated'] = acts_out.clone()
    return acts_out

def cache_recon_hook_with_abl(recon, hook):
    cache_with_ablation['recon'] = recon.clone().detach()
    
    acts_out = cache_with_ablation['in'].clone()
    # acts_out = t.zeros_like(cache_with_ablation['in'])
    acts_out[:, start_position:end_position, :] = (
        cache_with_ablation['in'][:, start_position:end_position, :] - 
        lambda_scale * recon[:, start_position:end_position, :]
    )
    cache_with_ablation['recon_ablated'] = acts_out.clone()
    return acts_out


sae.use_error_term = True
lambda_scale = 1.0
cache_no_ablation = {}
cache_with_ablation = {}

print("\n--- Running WITHOUT ablation ---")
# Run without ablation
model.add_sae(sae)
sae.use_error_term = True
model.add_hook(hook_sae_input, cache_input_hook_no_abl)
model.add_hook(hook_sae_acts_post, cache_acts_hook_no_abl)
model.add_hook(hook_sae_output, cache_output_hook_no_abl)
model.add_hook(hook_sae_recons, cache_recon_hook_no_abl)
sae.use_error_term = True
with t.no_grad():
    output_no_abl = model(tokens.input_ids)
    output_tokens_no_abl = model.generate(
        tokens.input_ids,
        # attention_mask=tokens.attention_mask,
        max_new_tokens=16,
        do_sample=False,
        freq_penalty=0,
        # num_beams=1,  
        # pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        temperature=0,
    )
model.reset_hooks(including_permanent=True)
model.reset_saes()
sae.use_error_term = True
print("\n--- Running WITH OUT ablation ---")
# Run with ablation
model.add_sae(sae)
sae.use_error_term = True
model.add_hook(hook_sae_input, cache_input_hook_with_abl)
model.add_hook(hook_sae_acts_post, cache_acts_hook_with_abl)
lambda_scale = 5.0
model.add_hook(hook_sae_output, cache_output_hook_with_abl)
lambda_scale = 0
model.add_hook(hook_sae_recons, cache_recon_hook_with_abl)
sae.use_error_term = True
with t.no_grad():
    output_with_abl = model(tokens.input_ids)
    output_tokens_with_abl = model.generate(
        tokens.input_ids,
        # attention_mask=tokens.attention_mask,
        max_new_tokens=16,
        do_sample=False,
        freq_penalty=0,
        # num_beams=1,  
        # pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        temperature=0,
    )
model.reset_hooks(including_permanent=True)
model.reset_saes()
print("Assertions WITH OUT ablation: ")
print("\n--- Assertions ---")
do_Assertions()
cache_with_ablation = {}
sae.use_error_term = True
print("\n--- Running WITH RECON ablation ---")
# Run with ablation
model.add_sae(sae)
sae.use_error_term = True
model.add_hook(hook_sae_input, cache_input_hook_with_abl)
model.add_hook(hook_sae_acts_post, cache_acts_hook_with_abl)
lambda_scale = 5.0
model.add_hook(hook_sae_recons, cache_recon_hook_with_abl)
lambda_scale = 0
model.add_hook(hook_sae_output, cache_output_hook_with_abl)
sae.use_error_term = True
with t.no_grad():
    output_with_abl = model(tokens.input_ids)
    output_tokens_with_recon_abl = model.generate(
        tokens.input_ids,
        # attention_mask=tokens.attention_mask,
        max_new_tokens=16,
        do_sample=False,
        freq_penalty=0,
        # num_beams=1,  
        # pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        temperature=0,
    )
model.reset_hooks(including_permanent=True)
model.reset_saes()
print("Assertions WITH RECON ablation: ")
print("\n--- Assertions ---")
do_Assertions()
cache_with_ablation = {}
sae.use_error_term = True

print("\n--- Running with latent manipulation but no ablation ---")
# Run with ablation
model.add_sae(sae)
sae.use_error_term = True
model.add_hook(hook_sae_input, cache_input_hook_with_abl)
model.add_hook(hook_sae_acts_post, cache_acts_hook_with_abl)
lambda_scale = 0
model.add_hook(hook_sae_output, cache_output_hook_with_abl)
model.add_hook(hook_sae_recons, cache_recon_hook_with_abl)
sae.use_error_term = True
with t.no_grad():
    output_with_abl = model(tokens.input_ids)
    output_tokens_with_no_out_manipulation = model.generate(
        tokens.input_ids,
        # attention_mask=tokens.attention_mask,
        max_new_tokens=16,
        do_sample=False,
        freq_penalty=0,
        # num_beams=1,  
        # pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        temperature=0,
    )
model.reset_hooks(including_permanent=True)
model.reset_saes()
print("Assertions in manipulation only case: ")
print("\n--- Assertions ---")
do_Assertions()
cache_with_ablation = {}
sae.use_error_term = True
# # get the 3 biggest indices in cache_with_ablation["feats"]
# biggest_indices = cache_with_ablation["feats"].topk(3, dim=-1).indices
# print(biggest_indices)
print(f"sae.use_error_term = {sae.use_error_term}")
print("generation results:")
print(model.to_string(output_tokens_no_abl))
print("\n")
print("Generation with OUT HOOK ablation:")
print(model.to_string(output_tokens_with_abl))
print("\n")
print("Generation with RECON HOOK ablation:")
print(model.to_string(output_tokens_with_recon_abl))
print("\n")
print("Generation with latent manipulation but NO OUT/RECON MANIPULATION:")
print(model.to_string(output_tokens_with_no_out_manipulation))

Tokenized shape: torch.Size([1, 8])

ABLATION TESTING
l5_acts shape: torch.Size([1, 8, 2304])

--- Running WITHOUT ablation ---


  0%|          | 0/16 [00:00<?, ?it/s]


--- Running WITH OUT ablation ---


  0%|          | 0/16 [00:00<?, ?it/s]

Assertions WITH OUT ablation: 

--- Assertions ---
sae.use_error_term = True

=== ABLATION ASSERTIONS ===
Assertion 1: Input is same for both cases
✅ Shapes match: torch.Size([1, 1, 2304])
✅ PASSED

Assertion 2: Features are same for both cases
✅ Shapes match: torch.Size([1, 1, 16384])
✅ PASSED

Assertion 3: Reconstruction is same for both cases
✅ Shapes match: torch.Size([1, 1, 2304])
❌ FAILED

Assertion 4: Output is same for both cases
✅ Shapes match: torch.Size([1, 1, 2304])
❌ FAILED

Assertion 5: Output is same as reconstruction for ablation case
✅ Shapes match: torch.Size([1, 1, 2304])
❌ FAILED

--- Running WITH RECON ablation ---


  0%|          | 0/16 [00:00<?, ?it/s]

Assertions WITH RECON ablation: 

--- Assertions ---
sae.use_error_term = True

=== ABLATION ASSERTIONS ===
Assertion 1: Input is same for both cases
✅ Shapes match: torch.Size([1, 1, 2304])
✅ PASSED

Assertion 2: Features are same for both cases
✅ Shapes match: torch.Size([1, 1, 16384])
✅ PASSED

Assertion 3: Reconstruction is same for both cases
✅ Shapes match: torch.Size([1, 1, 2304])
❌ FAILED

Assertion 4: Output is same for both cases
✅ Shapes match: torch.Size([1, 1, 2304])
❌ FAILED

Assertion 5: Output is same as reconstruction for ablation case
✅ Shapes match: torch.Size([1, 1, 2304])
❌ FAILED

--- Running with latent manipulation but no ablation ---


  0%|          | 0/16 [00:00<?, ?it/s]

Assertions in manipulation only case: 

--- Assertions ---
sae.use_error_term = True

=== ABLATION ASSERTIONS ===
Assertion 1: Input is same for both cases
✅ Shapes match: torch.Size([1, 1, 2304])
✅ PASSED

Assertion 2: Features are same for both cases
✅ Shapes match: torch.Size([1, 1, 16384])
✅ PASSED

Assertion 3: Reconstruction is same for both cases
✅ Shapes match: torch.Size([1, 1, 2304])
❌ FAILED

Assertion 4: Output is same for both cases
✅ Shapes match: torch.Size([1, 1, 2304])
❌ FAILED

Assertion 5: Output is same as reconstruction for ablation case
✅ Shapes match: torch.Size([1, 1, 2304])
❌ FAILED
sae.use_error_term = True
generation results:
['<bos>15+20= 35\n\n15+20=35\n\n15+2']


Generation with OUT HOOK ablation:
['<bos>15+20= 35\n\n15+20=35\n\n15+2']


Generation with RECON HOOK ablation:
['<bos>15+20= 35\n\n15+20=35\n\n15+2']


Generation with latent manipulation but NO OUT/RECON MANIPULATION:
['<bos>15+20= 35\n\n15+20=35\n\n15+2']


In [16]:
import torch
from transformers import AutoTokenizer
from sae_lens import HookedSAETransformer, SAE

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

# # Load model + SAE
# model_name = "google/gemma-2-2b-it"
# layer = 5
# model = HookedSAETransformer.from_pretrained(model_name, device=device)
# sae, _, _ = SAE.from_pretrained(
#     release="gemma-scope-2b-pt-res-canonical",
#     sae_id=f"layer_{layer}/width_16k/canonical",
#     device=str(device),
# )
# sae.use_error_term = False   # critical: disables error passthrough
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define hook names
hook_base = sae.cfg.hook_name
hook_output = f"{hook_base}.hook_sae_output"
hook_recons = f"{hook_base}.hook_sae_recons"
sae.use_error_term = True
# Test prompt
prompt = "Answer Directly 15+20= "
tokens = tokenizer([prompt], return_tensors="pt").to(device)

def ablation_hook(x, hook):
    # zero out the whole tensor as a simple ablation
    return torch.zeros_like(x)

def run_with_hook(hook_name):
    model.reset_hooks(including_permanent=True)
    model.reset_saes()
    model.add_sae(sae)

    if hook_name is not None:
        model.add_hook(hook_name, ablation_hook)
    with torch.no_grad():
        out_tokens = model.generate(
            tokens.input_ids,
            max_new_tokens=16,
            do_sample=False,
            temperature=0,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_tokens[0])

# Run three cases
baseline = run_with_hook(None)  # no ablation
out_ablation = run_with_hook(hook_output)
recon_ablation = run_with_hook(hook_recons)

print("=== Baseline ===")
print(baseline)
print("\n=== Output hook ablation ===")
print(out_ablation)
print("\n=== Recon hook ablation ===")
print(recon_ablation)


  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

=== Baseline ===
<bos>Answer Directly 15+20= 35

15+20=35

15+2

=== Output hook ablation ===
<bos>Answer Directly 15+20= <pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>

=== Recon hook ablation ===
<bos>Answer Directly 15+20= 



 to



  to 

line   

 


In [17]:
# Define hook names
hook_base = sae.cfg.hook_name
hook_output = f"{hook_base}.hook_sae_output"
hook_recons = f"{hook_base}.hook_sae_recons"

sae.use_error_term = False # critical: disables error passthrough

# Test prompt
prompt = "Answer Directly 15+20= "
tokens = tokenizer([prompt], return_tensors="pt").to(device)

def ablation_hook(x, hook):
    # zero out the whole tensor as a simple ablation
    return torch.zeros_like(x)

def run_with_hook(hook_name):
    model.reset_hooks(including_permanent=True)
    model.reset_saes()
    model.add_sae(sae)

    if hook_name is not None:
        model.add_hook(hook_name, ablation_hook)
    with torch.no_grad():
        out_tokens = model.generate(
            tokens.input_ids,
            max_new_tokens=16,
            do_sample=False,
            temperature=0,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_tokens[0])

# Run three cases
baseline = run_with_hook(None)  # no ablation
out_ablation = run_with_hook(hook_output)
recon_ablation = run_with_hook(hook_recons)

print("=== Baseline ===")
print(baseline)
print("\n=== Output hook ablation ===")
print(out_ablation)
print("\n=== Recon hook ablation ===")
print(recon_ablation)


  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

=== Baseline ===
<bos>Answer Directly 15+20= 35+40= 75+70= 14

=== Output hook ablation ===
<bos>Answer Directly 15+20= <pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>

=== Recon hook ablation ===
<bos>Answer Directly 15+20= <pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
